# Knee MRI diagnosis with DINOv2

**Standalone notebook · full original dataset · one NVIDIA RTX 5090**

Everything specific to this training pipeline is defined in the cells below:
input checks, DICOM reading, preprocessing, the dataset, the complete study
model, loss, optimization, checkpoint recovery, evaluation and plots. You can
copy this single notebook elsewhere and run it without the project package,
its Python files, shell scripts or configuration files.

The notebook uses standard scientific Python libraries. The public DINOv2
ViT-S/14 image backbone comes from **timm**, just as a convolution layer comes
from PyTorch; all knee-specific model classes and functions are visible below.
The first use downloads the authors' pretrained weights and verifies their
tensor fingerprint. These weights are general-image pretraining, not a knee
MRI model.

You still need your original MRI files, report-label export, frozen scanner
split and series-policy JSON. These are data inputs, not executable code. The
notebook preserves their existing train/validation boundary and trains all
permitted studies. Validation and expert studies never produce gradients.

Read the explanation, run each definition cell, then run the workflow cells at
the end. This starts a separate twelve-epoch run in its own output directory.

## 1. Environment and paths

Use the Python environment whose PyTorch and torchvision already work on the
5090. If needed, install the following packages in that environment **before**
starting Jupyter (there is no editable project installation):

    python -m pip install "timm==1.0.20" "numpy>=1.26" "pandas>=2" "scikit-learn>=1.3" "matplotlib>=3.8" "pydicom>=3" "python-gdcm>=3.0.10" "jupyterlab>=4,<5" "tornado==6.5.8" "ipykernel>=6"

The temporary Tornado pin supports the local Jupyter static-file handler.
Launch Jupyter bound to 127.0.0.1 and keep its normal authentication enabled.
The GPU check below performs real arithmetic and backpropagation.

**WORK_ROOT** is just a directory containing your data/artifacts and output;
it does not need to contain any project source. The optional paths can be set
explicitly. With the defaults, discovery looks only inside WORK_ROOT/runs,
first for input paths recorded in an existing frozen data protocol, then for
matching data files. Ambiguous matches stop and ask for an explicit path.

For portable Jupyter execution, the DataLoader uses **zero subprocess
workers**. Notebook-defined classes therefore need no exported worker module.
This can reduce data-loading throughput; it does not change the sampling,
model, loss, batch size, or split. GPU micro-batches remain two triplets and
optimizer batches remain two studies.

In [ ]:
from pathlib import Path

WORK_ROOT = Path("/media/talafha/Disk_1/CNN_CPC")
DATA_ROOT = WORK_ROOT / "rsna-knee-abnormality-detection"
RUN_ROOT = WORK_ROOT / "runs/dinov2_knee_mri_standalone"

# Set these to existing data artifacts if automatic discovery is ambiguous.
LABELS_ROOT = None
SCANNER_SPLIT_ROOT = None
SERIES_POLICY = None
DEVICE = "cuda:0"

In [ ]:
from __future__ import annotations

import ast
from contextlib import contextmanager, nullcontext
from dataclasses import dataclass
from pathlib import Path
import gc
import hashlib
import inspect
import json
import math
import os
import random
import sys
import time
from textwrap import dedent
from typing import Any, Iterable, Sequence
import warnings

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd
import pydicom
from sklearn.metrics import roc_auc_score
import timm
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
import torchvision
from IPython.display import display

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_TARGETS = len(TARGETS)
FORMAT_VERSION = "standalone_dinov2_knee_v1"
DINO_MODEL = "vit_small_patch14_dinov2.lvd142m"
DINO_URL = "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth"
PUBLIC_TENSOR_SHA256 = "da01dc066b2a6aea10165a09242353c4c9178032d76f9a5494e880cf5864dd92"
SERIES_SIGNATURE = "5c4bb1c52294e45f9e83274c5c07d198dc54811c49b96111b7c8439bd7bcd376"

RECIPE = dict(seed=2026, epochs=12, batch_size=2, encoder_lr=1e-5, head_lr=1e-4,
              weight_decay=1e-4, grad_clip=1.0, encoder_chunk_size=2,
              gradient_checkpointing=True, dim=384, heads=6, dropout=0.1,
              slices_per_series=32, base_slices=16, triplet_gap=1, crop_fraction=0.90,
              reference_area=448**2, alignment=32, augmentation=False,
              slice_layers=1, slice_position_embedding=False, auxiliary_loss=0.0,
              selection="fixed_final_epoch", tta=False)
LABEL_RULE = dict(min_confidence=0.75, positive_target=0.85, negative_target=0.05,
                  positive_weight=0.50, negative_weight=1.00)
EXPECTED_POPULATION = dict(report_only=4349, usable_cells=34010, train=3603,
                           validation=548, excluded_profile_overlap=198, expert=58)
VALIDATION_NAME = "validation_unseen_scanners"

## 2. File integrity, run ownership and environment checks

A run records input hashes, the actual notebook function definitions, the
recipe, library versions and public initialization. Resume requires the same
contract. A changed definition or input stops resume instead of mixing runs.
Checkpoint writes are atomic. On Linux, an exclusive file lock prevents two
notebooks from writing the same run simultaneously.

An interrupted epoch is repeated from the last completed epoch. The notebook
uses its own checkpoint format; choose its dedicated output folder rather
than an older training folder. Ordinary kernel interruption releases the GPU
model in a finally block. After a forced kernel shutdown, restart the kernel
and rerun the cells with the same settings.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False, separators=(',', ':')).encode()).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.writing')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n')
    os.replace(temporary, path)


def coverage(target, weight):
    result = {}
    for j, name in enumerate(TARGETS):
        active = weight[:, j] > 0
        result[name] = {'positive': int((active & (target[:, j] > 0.5)).sum()), 'negative': int((active & (target[:, j] < 0.5)).sum()), 'unlabelled': int((~active).sum())}
    return result


def require_same_run(saved, expected):
    if saved != expected:
        keys = sorted((k for k in set(saved) | set(expected) if saved.get(k) != expected.get(k)))
        raise ValueError(f'established recipe resume/comparison contract mismatch: {keys}')

In [ ]:
def atomic_torch_save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".writing")
    torch.save(value, temporary)
    os.replace(temporary, path)


def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".writing")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def finite_json(value):
    if isinstance(value, dict):
        return {k: finite_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [finite_json(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value


def notebook_code_digest():
    """Hash executed definitions, independent of notebook name/cell line numbers."""
    definitions = {}
    for name in IMPLEMENTATION_NAMES:
        obj = globals()[name]
        if inspect.isclass(obj):
            methods = {}
            for key, member in vars(obj).items():
                if isinstance(member, (staticmethod, classmethod)):
                    member = member.__func__
                if inspect.isfunction(member):
                    member = inspect.unwrap(member)
                    # Dataclass-generated methods have no user-written source.
                    if member.__code__.co_filename == "<string>":
                        continue
                    methods[key] = ast.dump(ast.parse(dedent(inspect.getsource(member))))
            definitions[name] = {"methods": methods,
                "annotations": {k: str(v) for k, v in getattr(obj, "__annotations__", {}).items()}}
        else:
            definitions[name] = ast.dump(ast.parse(dedent(inspect.getsource(inspect.unwrap(obj)))))
    return digest(definitions)


def implementation_contract():
    expected = dict(slices_per_series=32, base_slices=16, triplet_gap=1, crop_fraction=0.90,
                    reference_area=448**2, alignment=32, augmentation=False,
                    slice_layers=1, slice_position_embedding=False, auxiliary_loss=0.0,
                    selection="fixed_final_epoch", tta=False)
    if any(RECIPE.get(k) != v for k, v in expected.items()):
        raise ValueError("The declared geometry/context must match the implemented fixed recipe.")
    return {"version": FORMAT_VERSION, "code_sha256": notebook_code_digest(),
            "recipe": RECIPE, "labels": LABEL_RULE, "targets": TARGETS,
            "expected_population": EXPECTED_POPULATION,
            "public_tensor_sha256": PUBLIC_TENSOR_SHA256,
            "series_signature": SERIES_SIGNATURE,
            "dicom_suffixes": sorted(DICOM_SUFFIXES), "planes": list(PLANES),
            "plane_ids": PLANE_TO_ID, "true_flags": sorted(TRUE_TOKENS),
            "false_flags": sorted(FALSE_TOKENS), "allowed_splits": sorted(ALLOWED_SPLITS),
            "validation_assignment": VALIDATION_NAME, "backbone": DINO_MODEL, "public_url": DINO_URL}


@contextmanager
def exclusive_run(root):
    root = Path(root).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    if os.name != "posix":
        raise RuntimeError("This local training notebook uses Linux/POSIX file locking.")
    import fcntl
    with (root / ".run.lock").open("a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError("Another process is writing this run folder.") from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


def claim_run(root):
    root = Path(root)
    manifest = root / "notebook_run.json"
    contract = implementation_contract()
    if manifest.exists():
        require_same_run(json.loads(manifest.read_text()), contract)
    else:
        occupied = [p for p in root.iterdir() if p.name != ".run.lock"]
        if occupied:
            raise FileExistsError("Choose an empty output directory; this one contains another run.")
        write_json(manifest, contract)


def runtime_for(device):
    device = torch.device(device)
    if device.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("No CUDA device. Select the environment with working 5090 PyTorch.")
        torch.cuda.set_device(device)
        amp_dtype = torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    elif device.type == "cpu":
        amp_dtype = None  # Only practical for the small synthetic verification suite.
    else:
        raise ValueError("Use a CUDA device for full-data training.")
    return device, amp_dtype


def autocast(device, amp_dtype):
    return torch.autocast(device_type=device.type, dtype=amp_dtype or torch.float32,
                          enabled=amp_dtype is not None)


def environment_check(device):
    if timm.__version__ != "1.0.20":
        raise RuntimeError("Install timm==1.0.20 before running this recipe.")
    device, amp_dtype = runtime_for(device)
    with torch.enable_grad():
        x = torch.ones((32, 32), device=device, requires_grad=True)
        with autocast(device, amp_dtype):
            loss = (x @ x).square().mean()
        loss.backward()
        if x.grad is None or not torch.isfinite(x.grad).all():
            raise RuntimeError("GPU arithmetic/backpropagation failed.")
    del x, loss
    from pydicom.pixels import get_decoder
    for name, uid in {"JPEG Lossless": "1.2.840.10008.1.2.4.57",
                      "JPEG Lossless SV1": "1.2.840.10008.1.2.4.70",
                      "JPEG2000 Lossless": "1.2.840.10008.1.2.4.90",
                      "JPEG2000": "1.2.840.10008.1.2.4.91"}.items():
        decoder = get_decoder(uid)
        if not decoder.is_available:
            raise RuntimeError(f"Missing {name} decoder: {decoder.missing_dependencies}")
    print("Python:", sys.executable)
    print("Device:", torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")
    print("Precision:", amp_dtype or torch.float32)
    print("CUDA arithmetic and DICOM decoder checks: PASS")

## 3. DICOM loading and scan metadata

Each study supplies all eligible sagittal, coronal and axial series. Known
metadata comes from the series CSV. Missing values are repaired from DICOM
orientation and acquisition timing using the existing deterministic rules.

Pixels are ordered by physical slice position when available, with an
instance-number fallback. Rescale slope/intercept and MONOCHROME1 inversion
are applied. For compatibility with the original loader, individual unreadable
files may be skipped; an entirely unreadable or missing series raises an error.
This is not a file-by-file completeness audit of every scan.

In [ ]:
DICOM_SUFFIXES = {"", ".dcm", ".dicom", ".ima"}
PLANES = ("Sagittal", "Coronal", "Axial")
PLANE_TO_ID = {"Sagittal": 1, "Coronal": 2, "Axial": 3}
TRUE_TOKENS = {"true", "t", "yes", "y", "1", "1.0"}
FALSE_TOKENS = {"false", "f", "no", "n", "0", "0.0"}

In [ ]:
def _sort_key(ds) -> float:
    try:
        return float(np.dot(np.asarray(ds.ImagePositionPatient, float), np.cross(np.asarray(ds.ImageOrientationPatient, float)[:3], np.asarray(ds.ImageOrientationPatient, float)[3:])))
    except Exception:
        return float(getattr(ds, 'InstanceNumber', 0))


def find_series_dir(root: str | Path, split: str, study: str, series: str) -> Path | None:
    root = Path(root)
    study, series = (str(study), str(series))
    for p in (root / f'{split}_series' / study / series, root / f'{split}_images' / study / series, root / study / series):
        if p.is_dir():
            return p
    return None


def _iter_dicom_files(path: Path) -> list[Path]:
    return sorted((p for p in path.iterdir() if p.is_file() and p.suffix.lower() in DICOM_SUFFIXES))


def _pixel_spacing(ds) -> tuple[float, float] | None:
    try:
        values = np.asarray(getattr(ds, 'PixelSpacing'), dtype=float).reshape(-1)
        if len(values) < 2 or not np.isfinite(values[:2]).all() or np.any(values[:2] <= 0):
            return None
        return (float(values[0]), float(values[1]))
    except Exception:
        return None


def _pad_or_crop(image: np.ndarray, target: tuple[int, int]) -> np.ndarray:
    out = np.zeros(target, dtype=image.dtype)
    rows = min(image.shape[0], target[0])
    cols = min(image.shape[1], target[1])
    sr, sc = ((image.shape[0] - rows) // 2, (image.shape[1] - cols) // 2)
    tr, tc = ((target[0] - rows) // 2, (target[1] - cols) // 2)
    out[tr:tr + rows, tc:tc + cols] = image[sr:sr + rows, sc:sc + cols]
    return out


def read_dicom_series(path: str | Path, *, return_stats: bool=False):
    import pydicom
    path = Path(path)
    candidates = _iter_dicom_files(path)
    items = []
    failed = 0
    spacings: list[tuple[float, float]] = []
    slice_thickness: list[float] = []
    manufacturers: list[str] = []
    models: list[str] = []
    field_strengths: list[float] = []
    for p in candidates:
        try:
            ds = pydicom.dcmread(str(p), force=True)
            arr = np.asarray(ds.pixel_array, dtype=np.float32)
            arr = arr * float(getattr(ds, 'RescaleSlope', 1.0)) + float(getattr(ds, 'RescaleIntercept', 0.0))
            if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
                arr = arr.max() - arr
            spacing = _pixel_spacing(ds)
            if spacing is not None:
                spacings.append(spacing)
            try:
                value = float(getattr(ds, 'SliceThickness'))
                if np.isfinite(value) and value > 0:
                    slice_thickness.append(value)
            except Exception:
                pass
            manufacturer = str(getattr(ds, 'Manufacturer', '')).strip()
            model = str(getattr(ds, 'ManufacturerModelName', '')).strip()
            if manufacturer:
                manufacturers.append(manufacturer)
            if model:
                models.append(model)
            try:
                value = float(getattr(ds, 'MagneticFieldStrength'))
                if np.isfinite(value) and value > 0:
                    field_strengths.append(value)
            except Exception:
                pass
            base = _sort_key(ds)
            if arr.ndim == 2:
                items.append((base, arr))
            elif arr.ndim == 3:
                items.extend(((base + i * 0.0001, frame) for i, frame in enumerate(arr)))
            else:
                raise RuntimeError(f'unsupported pixel shape {arr.shape}')
        except Exception:
            failed += 1
    if not items:
        raise RuntimeError(f'No readable DICOM pixels in {path} ({len(candidates)} candidates, {failed} failures)')
    items.sort(key=lambda x: x[0])
    frames = [f for _, f in items]
    shapes = {f.shape for f in frames}
    if len(shapes) > 1:
        target = max(shapes, key=lambda s: s[0] * s[1])
        frames = [_pad_or_crop(f, target) for f in frames]
    volume = np.stack(frames).astype(np.float32, copy=False)
    if not return_stats:
        return volume
    spacing = None
    spacing_spread = None
    if spacings:
        arr = np.asarray(spacings, dtype=float)
        spacing = [float(x) for x in np.median(arr, axis=0)]
        spacing_spread = [float(x) for x in np.ptp(arr, axis=0)]
    stats = {'candidate_files': len(candidates), 'decode_failures': failed, 'decoded_frames': len(frames), 'pixel_spacing_mm': spacing, 'pixel_spacing_range_mm': spacing_spread, 'slice_thickness_mm': float(np.median(slice_thickness)) if slice_thickness else None, 'manufacturer': manufacturers[0] if manufacturers else None, 'manufacturer_model': models[0] if models else None, 'magnetic_field_strength_t': float(np.median(field_strengths)) if field_strengths else None, 'rows': int(volume.shape[1]), 'columns': int(volume.shape[2])}
    if spacing is not None:
        stats['fov_mm'] = [float(volume.shape[1] * spacing[0]), float(volume.shape[2] * spacing[1])]
    else:
        stats['fov_mm'] = None
    return (volume, stats)


def _normalise_volume(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32)
    if v.ndim != 3 or len(v) == 0:
        raise RuntimeError(f'expected non-empty [S,H,W], got {v.shape}')
    finite = v[np.isfinite(v)]
    if finite.size == 0:
        raise RuntimeError('DICOM volume contains no finite pixels')
    lo, hi = np.percentile(finite, [1, 99])
    v = np.nan_to_num(v, nan=float(lo), posinf=float(hi), neginf=float(lo))
    v = np.clip(v, lo, hi)
    return ((v - lo) / max(float(hi - lo), 1e-06)).astype(np.float32, copy=False)


def _centers(n_frames: int, n_slices: int, gap: int, *, center_offset: int=0, jitter: int=0, rng: np.random.Generator | None=None) -> np.ndarray:
    """Distributed center positions, avoiding edge duplication whenever possible."""
    if n_frames < 1:
        raise ValueError('n_frames must be positive')
    lo, hi = (gap, n_frames - 1 - gap) if n_frames > 2 * gap else (0, n_frames - 1)
    centers = np.round(np.linspace(lo, hi, n_slices)).astype(int) + int(center_offset)
    if jitter > 0:
        rng = rng or np.random.default_rng()
        centers += rng.integers(-int(jitter), int(jitter) + 1, size=n_slices)
    return np.clip(centers, 0, n_frames - 1)

In [ ]:
def plane_from_orientation(orientation: Sequence[float] | None) -> str | None:
    """Classify the imaging plane from `ImageOrientationPatient`.

    The tag holds the row and column direction cosines. Their cross product is
    the slice normal, and whichever patient axis it aligns with most strongly
    names the plane: x is sagittal, y coronal, z axial.

    Returns ``None`` when the tag is missing or degenerate, so callers can tell
    "unknown" apart from a genuine answer.
    """
    if orientation is None or len(orientation) < 6:
        return None
    row = np.asarray(orientation[:3], dtype=np.float64)
    column = np.asarray(orientation[3:6], dtype=np.float64)
    normal = np.cross(row, column)
    norm = float(np.linalg.norm(normal))
    if norm < 1e-06:
        return None
    return PLANES[int(np.argmax(np.abs(normal / norm)))]


def weighting_from_parameters(echo_time: float | None, repetition_time: float | None, inversion_time: float | None=None, scan_options: str='', image_type: Sequence[str] | None=None) -> tuple[str, bool]:
    """Infer contrast weighting and fat suppression from acquisition timings.

    Returns a ``(weighting, fat_suppressed)`` pair, where weighting is one of
    ``t1``, ``pd``, ``t2``, ``stir`` or ``unknown``. The thresholds are the
    conventional musculoskeletal ones: short TE with short TR is T1, short TE
    with long TR is proton density, long TE with long TR is T2.
    """
    haystack = ' '.join([str(scan_options or '')] + [str(value) for value in image_type or []]).upper()
    fat_suppressed = any((token in haystack for token in ('FS', 'FAT_SAT', 'FATSAT', 'SPAIR', 'SPIR', 'DIXON', 'STIR')))
    if inversion_time is not None and 0 < float(inversion_time) < 200:
        return ('stir', True)
    if echo_time is None or repetition_time is None:
        return ('unknown', fat_suppressed)
    te = float(echo_time)
    tr = float(repetition_time)
    if te < 35.0:
        return ('t1' if tr < 900.0 else 'pd', fat_suppressed)
    if tr >= 900.0:
        return ('t2', fat_suppressed)
    return ('unknown', fat_suppressed)


def is_fluid_sensitive(weighting: str) -> bool:
    """Whether a weighting shows fluid brightly.

    T2, proton density and STIR are fluid sensitive; T1 is structural. This
    mirrors the meaning of the `Fluid_Sensitive` column in `train_series.csv`.
    """
    return weighting in {'t2', 'pd', 'stir'}


def _multiframe_orientation(dataset: Any) -> Sequence[float] | None:
    """Recover orientation from an enhanced multi-frame instance."""
    try:
        shared = dataset.SharedFunctionalGroupsSequence[0]
        return shared.PlaneOrientationSequence[0].ImageOrientationPatient
    except Exception:
        return None


def read_series_metadata(series_dir: str | Path) -> dict[str, Any]:
    """Read one DICOM from a series directory and describe the series.

    Only a single instance is opened — every slice of a series shares these
    attributes, so reading more would cost time for nothing.

    Returns a dict with ``Anatomical_Plane``, ``Fluid_Sensitive``,
    ``Fat_Suppression`` and ``weighting``. Values are ``None`` when they cannot
    be determined, so a caller can leave the CSV value in place.
    """
    import pydicom
    unknown: dict[str, Any] = {'Anatomical_Plane': None, 'Fluid_Sensitive': None, 'Fat_Suppression': None, 'weighting': None}
    path = Path(series_dir)
    if not path.is_dir():
        return unknown
    dataset = None
    for candidate in sorted(path.iterdir()):
        if not candidate.is_file():
            continue
        try:
            dataset = pydicom.dcmread(str(candidate), force=True, stop_before_pixels=True)
            break
        except Exception:
            continue
    if dataset is None:
        return unknown
    orientation = getattr(dataset, 'ImageOrientationPatient', None)
    if orientation is None:
        orientation = _multiframe_orientation(dataset)
    weighting, fat_suppressed = weighting_from_parameters(getattr(dataset, 'EchoTime', None), getattr(dataset, 'RepetitionTime', None), getattr(dataset, 'InversionTime', None), str(getattr(dataset, 'ScanOptions', '') or ''), getattr(dataset, 'ImageType', []) or [])
    return {'Anatomical_Plane': plane_from_orientation(orientation), 'Fluid_Sensitive': None if weighting == 'unknown' else is_fluid_sensitive(weighting), 'Fat_Suppression': bool(fat_suppressed), 'weighting': None if weighting == 'unknown' else weighting}

In [ ]:
def _require_columns(df: pd.DataFrame, required: set[str], name: str) -> None:
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')


def load_train_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    _require_columns(df, {'StudyInstanceUID', 'Report', *TARGETS}, 'train.csv')
    if df['StudyInstanceUID'].isna().any():
        raise ValueError('train.csv contains missing StudyInstanceUID values')
    df = df.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    if df['StudyInstanceUID'].duplicated().any():
        raise ValueError('train.csv contains duplicate StudyInstanceUID values')
    return df


def gold_mask(df: pd.DataFrame) -> pd.Series:
    return df[TARGETS].notna().any(axis=1)


def coerce_bool(values: pd.Series, *, preserve_unknown: bool=False) -> pd.Series:
    """Parse metadata flags without converting missing/unknown values to True.

    ``preserve_unknown=True`` returns pandas' nullable Boolean dtype so DICOM
    metadata can repair unknown sequence flags before routing. The public helper
    keeps the historical conservative default of mapping unknown values to False.
    """
    result = pd.Series(pd.NA, index=values.index, dtype='boolean')
    if pd.api.types.is_bool_dtype(values):
        result.loc[values.notna()] = values.loc[values.notna()].astype(bool)
    elif pd.api.types.is_numeric_dtype(values):
        known = values.notna()
        result.loc[known] = values.loc[known].astype(float).ne(0.0)
    else:
        text = values.astype('string').str.strip().str.lower()
        result.loc[text.isin(TRUE_TOKENS)] = True
        result.loc[text.isin(FALSE_TOKENS)] = False
    return result if preserve_unknown else result.fillna(False).astype(bool)


def normalise_plane(values: pd.Series) -> pd.Series:
    mapping = {'sagittal': 'Sagittal', 'sag': 'Sagittal', 'sagital': 'Sagittal', 'coronal': 'Coronal', 'cor': 'Coronal', 'axial': 'Axial', 'ax': 'Axial', 'transverse': 'Axial'}
    text = values.astype('string').str.strip().str.lower()
    return text.map(mapping).fillna('').astype(str)


def load_series_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'StudyInstanceUID', 'SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane'}
    _require_columns(df, required, 'series CSV')
    if df[['StudyInstanceUID', 'SeriesInstanceUID']].isna().any().any():
        raise ValueError('series CSV contains missing study/series UID values')
    df = df.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)
    df['Fluid_Sensitive'] = coerce_bool(df['Fluid_Sensitive'], preserve_unknown=True)
    df['Fat_Suppression'] = coerce_bool(df['Fat_Suppression'], preserve_unknown=True)
    df['Anatomical_Plane'] = normalise_plane(df['Anatomical_Plane'])
    if df[['StudyInstanceUID', 'SeriesInstanceUID']].duplicated().any():
        raise ValueError('series CSV contains duplicate study/series rows')
    return df


def backfill_series_metadata(series_df: pd.DataFrame, data_root: str | Path, split: str='train', limit: int | None=None) -> tuple[pd.DataFrame, dict[str, int]]:
    """Independently repair missing plane, fluid and fat-suppression metadata."""
    df = series_df.copy()
    missing_plane = df['Anatomical_Plane'].astype(str).str.strip().eq('')
    missing_fluid = df['Fluid_Sensitive'].isna()
    missing_fat = df['Fat_Suppression'].isna()
    needs = missing_plane | missing_fluid | missing_fat
    targets = df.index[needs]
    if limit is not None:
        targets = targets[:int(limit)]
    stats = {'rows_needing_metadata': int(needs.sum()), 'missing_plane': int(missing_plane.sum()), 'missing_fluid': int(missing_fluid.sum()), 'missing_fat_suppression': int(missing_fat.sum()), 'inspected': len(targets), 'repaired_plane': 0, 'repaired_fluid': 0, 'repaired_fat_suppression': 0}
    for index in targets:
        row = df.loc[index]
        series_dir = find_series_dir(data_root, split, str(row['StudyInstanceUID']), str(row['SeriesInstanceUID']))
        if series_dir is None:
            continue
        metadata = read_series_metadata(series_dir)
        if missing_plane.loc[index] and metadata['Anatomical_Plane'] is not None:
            df.at[index, 'Anatomical_Plane'] = metadata['Anatomical_Plane']
            stats['repaired_plane'] += 1
        if missing_fluid.loc[index] and metadata['Fluid_Sensitive'] is not None:
            df.at[index, 'Fluid_Sensitive'] = bool(metadata['Fluid_Sensitive'])
            stats['repaired_fluid'] += 1
        if missing_fat.loc[index] and metadata['Fat_Suppression'] is not None:
            df.at[index, 'Fat_Suppression'] = bool(metadata['Fat_Suppression'])
            stats['repaired_fat_suppression'] += 1
    return (df, stats)

In [ ]:
def _flag_id(value) -> int:
    """0=unknown, 1=false, 2=true."""
    if pd.isna(value):
        return 0
    return 2 if bool(value) else 1


def build_variable_series_index(series_df: pd.DataFrame, studies: Iterable[str]) -> dict[str, list[dict]]:
    """Return every repaired series with a recognized anatomical plane.

    No fluid/structural winner is selected. Ordering is deterministic only for
    reproducibility; the established recipe model has no series-position embedding.
    """
    work = series_df.copy()
    work['StudyInstanceUID'] = work['StudyInstanceUID'].astype(str)
    work['SeriesInstanceUID'] = work['SeriesInstanceUID'].astype(str)
    grouped = {uid: part for uid, part in work.groupby('StudyInstanceUID', sort=False)}
    empty = work.iloc[0:0]
    result: dict[str, list[dict]] = {}
    for study in studies:
        uid = str(study)
        records: list[dict] = []
        part = grouped.get(uid, empty)
        for _, row in part.iterrows():
            plane = str(row.get('Anatomical_Plane', ''))
            plane_id = PLANE_TO_ID.get(plane, 0)
            if plane_id == 0:
                continue
            records.append({'series_uid': str(row['SeriesInstanceUID']), 'plane': plane, 'plane_id': int(plane_id), 'fluid_id': _flag_id(row.get('Fluid_Sensitive')), 'fat_id': _flag_id(row.get('Fat_Suppression'))})
        records.sort(key=lambda x: (int(x['plane_id']), int(x['fluid_id']), int(x['fat_id']), str(x['series_uid'])))
        result[uid] = records
    return result

## 4. Labels and the frozen scanner boundary

Report states with confidence at least **0.75** supervise training. Positive
states use target **0.85** and weight **0.5**; negated states use target **0.05**
and weight **1.0**. Missing or uncertain cells have weight zero. These soft
targets describe the established training recipe, not calibrated probabilities.

The original report-only population is 4,349 studies with 34,010 supervised
cells. The scanner split yields 3,603 training studies, 548 validation studies
and 198 additional excluded studies whose scanner profiles overlap validation.
The 58 expert studies are a diagnostic evaluation set only.

Scanner separation is enforced across the complete training population.
Patient identities are unavailable in this contract, so this is not a claim
of verified patient separation. The reused validation surface is a development
measure, not an untouched test or competition score.

In [ ]:
def load_fill_merged_export(root: str | Path) -> tuple[pd.DataFrame, dict, dict]:
    """Read a fill-only merged export and insist it overrode nothing.

    The merge is usable precisely because it preserves every parser call, which
    is what keeps the frozen specificity intact. An export claiming otherwise is
    a different experiment and must not be trained on under this name.
    """
    root = Path(root)
    targets_path = root / 'training_targets.csv'
    policy_path = root / 'policy.json'
    audit_path = root / 'audit.json'
    for path in (targets_path, policy_path, audit_path):
        if not path.is_file():
            raise FileNotFoundError(f'fill-merged export is missing {path}')
    policy = json.loads(policy_path.read_text(encoding='utf-8'))
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    if int(audit.get('base_cells_overridden', -1)) != 0:
        raise ValueError('this export overrode base parser cells; fill-only is what makes the frozen specificity carry through, so it is required here')
    if int(audit.get('gold_rows_in_training_targets', -1)) != 0:
        raise ValueError('fill-merged export does not certify zero gold rows')
    return (pd.read_csv(targets_path), policy, audit)


def prepare_all_report_only_supervision(train_df: pd.DataFrame, supervision_frame: pd.DataFrame) -> tuple[list[str], np.ndarray, np.ndarray, dict]:
    """Build soft targets and weights, retaining zero-weight studies."""
    train = train_df.copy()
    train['StudyInstanceUID'] = train['StudyInstanceUID'].astype(str)
    frame = supervision_frame.copy()
    frame['StudyInstanceUID'] = frame['StudyInstanceUID'].astype(str)
    if frame['StudyInstanceUID'].duplicated().any():
        raise ValueError('supervision table contains duplicate StudyInstanceUID values')
    gold = gold_mask(train)
    gold_uids = set(train.loc[gold, 'StudyInstanceUID'])
    overlap = gold_uids.intersection(set(frame['StudyInstanceUID']))
    if overlap:
        raise ValueError(f'Report-label supervision contains {len(overlap)} gold UID(s)')
    non_gold = train.loc[~gold, ['StudyInstanceUID']].copy()
    if len(non_gold) != EXPECTED_POPULATION['report_only']:
        raise ValueError('Report-label requires exactly 4,349 report-only studies')
    expected = set(non_gold['StudyInstanceUID'])
    actual = set(frame['StudyInstanceUID'])
    if expected != actual:
        raise ValueError(f'Report-label report-only UID mismatch: missing={len(expected - actual)}, extra={len(actual - expected)}')
    ordered = non_gold.merge(frame, on='StudyInstanceUID', how='left', validate='one_to_one')
    n = len(ordered)
    y = np.full((n, len(TARGETS)), 0.5, dtype=np.float32)
    w = np.zeros((n, len(TARGETS)), dtype=np.float32)
    per_target: dict[str, dict] = {}
    for j, target in enumerate(TARGETS):
        state_col = f'{target}__state'
        conf_col = f'{target}__confidence'
        if state_col not in ordered or conf_col not in ordered:
            raise ValueError(f'supervision table missing state/confidence for {target}')
        state = ordered[state_col].fillna('').astype(str).to_numpy()
        conf = pd.to_numeric(ordered[conf_col], errors='coerce').fillna(0.0).to_numpy(float)
        positive = (state == 'positive') & (conf >= LABEL_RULE['min_confidence'])
        negative = (state == 'negated') & (conf >= LABEL_RULE['min_confidence'])
        y[positive, j] = LABEL_RULE['positive_target']
        y[negative, j] = LABEL_RULE['negative_target']
        w[positive, j] = LABEL_RULE['positive_weight']
        w[negative, j] = LABEL_RULE['negative_weight']
        per_target[target] = {'positive_cells': int(positive.sum()), 'negative_cells': int(negative.sum()), 'usable_cells': int((positive | negative).sum()), 'base_weight_sum': float(w[:, j].sum())}
    active = w.sum(axis=1) > 0
    summary = {'report_only_rows': n, 'active_studies': int(active.sum()), 'inactive_studies_zero_usable_cells': int((~active).sum()), 'usable_cells': int((w > 0).sum()), 'positive_cells': int(((w > 0) & (y > 0.5)).sum()), 'negative_cells': int(((w > 0) & (y < 0.5)).sum()), 'targets': per_target, 'zero_weight_studies_retained_in_mri_exposure': True}
    return (ordered['StudyInstanceUID'].astype(str).tolist(), y, w, summary)

In [ ]:
def verify_selection_split(rows: pd.DataFrame) -> None:
    """Verify the frozen scanner assignments and their parent boundary."""
    required = {'StudyInstanceUID', 'scanner_profile', 'parent_split', 'selection_split'}
    missing = sorted(required.difference(rows.columns))
    if missing:
        raise ValueError(f'established recipe selection split rows are missing columns: {missing}')
    if rows['StudyInstanceUID'].astype(str).duplicated().any():
        raise ValueError('established recipe selection split contains duplicate study UIDs')
    labels = set(rows['selection_split'].astype(str))
    unknown = sorted(labels.difference(ALLOWED_SPLITS))
    if unknown:
        raise ValueError(f'established recipe selection split contains unknown labels: {unknown}')
    parent = rows['parent_split'].astype(str)
    assignment = rows['selection_split'].astype(str)
    prior = ~parent.eq('train')
    if not assignment.loc[prior].eq('excluded_prior_surface').all():
        raise ValueError('established recipe would reuse a earlier validation row instead of excluding it')
    if assignment.loc[~prior].eq('excluded_prior_surface').any():
        raise ValueError('established recipe left a parent-training row outside its fresh split')
    train_profiles = set(rows.loc[assignment.eq('train'), 'scanner_profile'].astype(str))
    seen_profiles = set(rows.loc[assignment.eq('validation_seen_scanners'), 'scanner_profile'].astype(str))
    unseen_profiles = set(rows.loc[assignment.eq(VALIDATION_NAME), 'scanner_profile'].astype(str))
    if not train_profiles or not seen_profiles or (not unseen_profiles):
        raise ValueError('established recipe requires non-empty train, seen, and unseen groups')
    if unseen_profiles.intersection(train_profiles) or unseen_profiles.intersection(seen_profiles):
        raise ValueError('established recipe unseen-scanner profiles straddle training or seen validation')
    if not seen_profiles.issubset(train_profiles):
        missing_profiles = sorted(seen_profiles.difference(train_profiles))
        raise ValueError(f'established recipe seen-scanner validation contains profiles absent from established recipe training: {missing_profiles[:5]}')

In [ ]:
def partition(uids, rows, expert_uids):
    """Exclude held scanner profiles from every report-only training row."""
    if len(uids) != len(set(uids)) or set(uids) & set(expert_uids):
        raise ValueError('duplicate report UIDs or expert overlap')
    needed = ['StudyInstanceUID', 'scanner_profile', 'selection_split']
    if not set(needed) <= set(rows) or rows[needed].isna().any().any():
        raise ValueError('split has missing UIDs, profiles or assignments')
    rows = rows.copy()
    rows['StudyInstanceUID'] = rows.StudyInstanceUID.astype(str)
    if rows.StudyInstanceUID.duplicated().any() or set(rows.StudyInstanceUID) != set(uids):
        raise ValueError('split/report UID population mismatch')
    rows = rows.set_index('StudyInstanceUID').loc[uids]
    profiles = rows.scanner_profile.astype(str).to_numpy()
    if any((not x.strip() for x in profiles)):
        raise ValueError('blank scanner profile')
    validation = rows.selection_split.eq(VALIDATION_NAME).to_numpy()
    held_profiles = set(profiles[validation])
    train = ~validation & ~np.isin(profiles, list(held_profiles))
    excluded = ~validation & ~train
    if not train.any() or not validation.any():
        raise ValueError('established recipe needs nonempty train and validation')
    return ({'train': np.flatnonzero(train), 'validation': np.flatnonzero(validation), 'excluded_profile_overlap': np.flatnonzero(excluded)}, profiles)

In [ ]:
ALLOWED_SPLITS = {"train", "validation_seen_scanners", "validation_unseen_scanners", "excluded_prior_surface"}


def only_match(paths, label):
    paths = sorted({Path(p).expanduser().resolve() for p in paths})
    if len(paths) != 1:
        choices = "\n".join(str(p) for p in paths[:10])
        raise FileNotFoundError(f"Set {label} explicitly: found {len(paths)} matching inputs.\n{choices}")
    return paths[0]


def load_json(path):
    return json.loads(Path(path).read_text())


def discover_inputs(work_root, labels_root=None, scanner_split_root=None, series_policy=None):
    search = Path(work_root) / "runs"
    recorded = {"training_targets": set(), "gate_json": set(), "series_policy": set()}
    for path in search.glob("**/protocol.json"):
        try:
            inputs = load_json(path)["inputs"]
            for key in recorded:
                item = inputs.get(key, {})
                candidate = Path(item.get("path", ""))
                if candidate.is_file() and sha256_file(candidate) == item.get("sha256"):
                    recorded[key].add(candidate.resolve())
        except (KeyError, OSError, ValueError):
            continue
    if labels_root is None:
        candidates = recorded["training_targets"]
        if not candidates:
            candidates = []
            for path in search.glob("**/training_targets.csv"):
                try:
                    audit = load_json(path.parent / "audit.json")
                    if (audit.get("base_cells_overridden") == 0
                            and audit.get("gold_rows_in_training_targets") == 0):
                        candidates.append(path)
                except (OSError, ValueError):
                    continue
        labels_root = only_match(candidates, "LABELS_ROOT").parent
    if scanner_split_root is None:
        candidates = recorded["gate_json"] or list(search.glob("**/*selection_split.json"))
        scanner_split_root = only_match(candidates, "SCANNER_SPLIT_ROOT").parent
    if series_policy is None:
        candidates = recorded["series_policy"]
        if not candidates:
            candidates = []
            for path in search.glob("**/series_policy.json"):
                try:
                    if load_json(path).get("series_summary", {}).get("series_signature_sha256") == SERIES_SIGNATURE:
                        candidates.append(path)
                except (OSError, ValueError):
                    continue
        series_policy = only_match(candidates, "SERIES_POLICY")
    return Path(labels_root).resolve(), Path(scanner_split_root).resolve(), Path(series_policy).resolve()


def canonical_split_rows(rows):
    """Accept historical column names as data, without hard-coding experiment labels."""
    rows = rows.copy()
    assignments = [c for c in rows if c.endswith("split") and not c.startswith("parent_")]
    parents = [c for c in rows if c.startswith("parent_") and c.endswith("split")]
    if len(assignments) != 1 or len(parents) != 1:
        raise ValueError("Expected one selection assignment and one parent assignment column.")
    rows = rows.rename(columns={assignments[0]: "selection_split", parents[0]: "parent_split"})
    needed = ["StudyInstanceUID", "scanner_profile", "selection_split", "parent_split"]
    _require_columns(rows, set(needed), "frozen split")
    if rows[needed].isna().any().any():
        raise ValueError("The frozen split contains missing assignments or identifiers.")
    verify_selection_split(rows)
    return rows


def validate_series_policy(path):
    policy = load_json(path)
    if (policy.get("policy") != "all_repaired_anatomical_series_v1"
            or policy.get("uses_gold_labels") is not False
            or policy.get("viability_passed") is not True
            or policy.get("series_summary", {}).get("series_signature_sha256") != SERIES_SIGNATURE):
        raise ValueError("The original label-free all-series policy is required.")
    for suffix, value in (("_active_studies", 3120), ("_usable_cells", 14123)):
        matches = [v for k, v in policy.items() if k.endswith(suffix)]
        if matches != [value]:
            raise ValueError("The series-policy population does not match the established input.")
    return policy


def prepare_data(*, work_root, data_root, run_root, labels_root=None,
                 scanner_split_root=None, series_policy=None):
    data_root, run_root = Path(data_root).resolve(), Path(run_root).resolve()
    if run_root in {data_root, Path(work_root).resolve(), Path(work_root).resolve() / "runs"}:
        raise ValueError("Use a dedicated run directory.")
    labels_root, gate, policy_path = discover_inputs(work_root, labels_root, scanner_split_root, series_policy)
    if gate.is_file():
        gate = gate.parent
    gate_json = only_match(gate.glob("*selection_split.json"), "SCANNER_SPLIT_ROOT")
    stem = gate_json.stem
    paths = dict(train_csv=data_root / "train.csv", train_series_csv=data_root / "train_series.csv",
                 training_targets=labels_root / "training_targets.csv", label_policy=labels_root / "policy.json",
                 label_audit=labels_root / "audit.json", series_policy=policy_path, gate_json=gate_json,
                 gate_rows=gate / f"{stem}_by_study.csv", gate_hash=gate / f"{stem}.sha256")
    inputs = {k: dict(path=str(p), sha256=sha256_file(p)) for k, p in paths.items()}
    out = run_root / "protocol"
    with exclusive_run(run_root):
        claim_run(run_root)
        if (out / "protocol.json").exists():
            protocol = load_protocol(run_root)
            if protocol["inputs"] != inputs or protocol["data_root"] != str(data_root):
                raise ValueError("Frozen data inputs changed; use the original inputs for resume.")
            return protocol
        if out.exists() and any(out.iterdir()):
            raise FileExistsError("Incomplete protocol exists. Inspect it before choosing a new run folder.")
        gate_payload = load_json(gate_json)
        if paths["gate_hash"].read_text().split()[0] != inputs["gate_json"]["sha256"]:
            raise ValueError("Scanner gate JSON/hash mismatch.")
        for source_key, input_key in (("source_train_csv_sha256", "train_csv"),
                                      ("source_training_targets_sha256", "training_targets")):
            if gate_payload.get(source_key) != inputs[input_key]["sha256"]:
                raise ValueError(f"Original gate/input mismatch: {source_key}")
        rows = canonical_split_rows(pd.read_csv(paths["gate_rows"], dtype={"StudyInstanceUID": str}))
        actual = set(rows.loc[rows.selection_split.eq(VALIDATION_NAME), "scanner_profile"].astype(str))
        if actual != set(gate_payload["unseen_scanner_profiles"]):
            raise ValueError("Gate JSON and CSV validation scanner profiles disagree.")
        train = load_train_csv(paths["train_csv"])
        frame, _, _ = load_fill_merged_export(labels_root)
        uids, target, weight, summary = prepare_all_report_only_supervision(train, frame)
        if summary["usable_cells"] != EXPECTED_POPULATION["usable_cells"]:
            raise ValueError("The original report-label export is required; supervised-cell count changed.")
        experts = train.loc[gold_mask(train)].copy()
        expert_uids = experts.StudyInstanceUID.tolist()
        indices, profiles = partition(uids, rows, expert_uids)
        splits = {name: [uids[int(i)] for i in ix] for name, ix in indices.items()}
        counts = {k: len(v) for k, v in splits.items()}
        if any(counts[k] != EXPECTED_POPULATION[k] for k in counts) or len(experts) != EXPECTED_POPULATION["expert"]:
            raise ValueError(f"Original full-data population changed: {counts}, expert={len(experts)}")
        audits = {name: coverage(target[ix], weight[ix]) for name, ix in indices.items()}
        for split in ("train", "validation"):
            if any(min(c["positive"], c["negative"]) == 0 for c in audits[split].values()):
                raise ValueError(f"Cannot measure all twelve targets in {split}.")
        validate_series_policy(policy_path)
        series = load_series_csv(paths["train_series_csv"])
        series, repair = backfill_series_metadata(series, data_root)
        index = build_variable_series_index(series, uids + expert_uids)
        for uid in uids + expert_uids:
            if not index[uid]:
                raise ValueError(f"No eligible MRI series for {uid}")
        for uid in splits["train"] + splits["validation"] + expert_uids:
            for record in index[uid]:
                directory = find_series_dir(data_root, "train", uid, record["series_uid"])
                if directory is None or not _iter_dicom_files(directory):
                    raise FileNotFoundError(f"Missing DICOM input: {uid}/{record['series_uid']}")
        out.mkdir(parents=True, exist_ok=True)
        write_json(out / "series_index.json", index)
        atomic_npz(out / "labels.npz", uids=np.asarray(uids), target=target, weight=weight,
                   expert_uids=np.asarray(expert_uids), expert_target=experts[TARGETS].to_numpy(np.float32))
        protocol = dict(implementation=implementation_contract(), data_root=str(data_root),
                        inputs=inputs, splits=splits, counts=counts, expert_uids=expert_uids,
                        split_uid_hashes={k: digest(v) for k, v in splits.items()},
                        scanner_profiles=dict(zip(uids, profiles.tolist())), coverage=audits,
                        metadata_repair=repair, competition_supervised_ancestor=False, gold_studies_in_gradient=0,
                        validation_role="reused development surface", expert_role="diagnostic only",
                        labels_sha256=sha256_file(out / "labels.npz"),
                        series_index_sha256=sha256_file(out / "series_index.json"))
        write_json(out / "protocol.json", protocol)
        (out / "protocol.sha256").write_text(sha256_file(out / "protocol.json") + "\n")
    return protocol


def load_protocol(run_root):
    root = Path(run_root) / "protocol"
    if sha256_file(root / "protocol.json") != (root / "protocol.sha256").read_text().strip():
        raise ValueError("Protocol hash mismatch.")
    p = load_json(root / "protocol.json")
    require_same_run(p["implementation"], implementation_contract())
    for name, key in (("labels.npz", "labels_sha256"), ("series_index.json", "series_index_sha256")):
        if sha256_file(root / name) != p[key]:
            raise ValueError(f"Frozen artifact changed: {name}")
    for key, item in p["inputs"].items():
        if sha256_file(item["path"]) != item["sha256"]:
            raise ValueError(f"Source input changed: {key}")
    return p

## 5. From a volume to 32 triplets

Intensity is normalized using the whole volume's 1st and 99th percentiles.
Sixteen base centers and sixteen additional centers sample the series.
Short volumes may repeat centers. Each input contains the preceding, center
and following slice; boundary indices are clamped.

A central **90%** crop is resized with one isotropic scale to approximately
**448² pixels**, then minimally reflection-padded to stride 32. Rectangular
series keep their aspect ratio. Each series becomes **[32, 3, H, W]**. There
are no random pixel augmentations and no test-time augmentation in this recipe.

In [ ]:
def _extra_centers(base: np.ndarray, dense: np.ndarray, count: int) -> np.ndarray:
    """Pick deterministic dense-grid centres not already used by the base path."""
    base_list = [int(x) for x in np.asarray(base).reshape(-1)]
    dense_list = [int(x) for x in np.asarray(dense).reshape(-1)]
    used = set(base_list)
    extras = [x for x in dense_list if x not in used]
    if len(extras) < count:
        for x in dense_list:
            extras.append(x)
            if len(extras) >= count:
                break
    while len(extras) < count:
        extras.append(base_list[len(extras) % len(base_list)])
    return np.asarray(extras[:count], dtype=np.int64)


def dense_centers(n_frames: int, *, gap: int=1, center_offset: int=0, base_slices: int=16, dense_slices: int=32) -> tuple[np.ndarray, np.ndarray]:
    """Return 32 centers: the fixed 16-center grid followed by 16 additional centers."""
    if dense_slices < base_slices:
        raise ValueError('dense_slices must be >= base_slices')
    base = _centers(n_frames, base_slices, gap, center_offset=center_offset, jitter=0)
    dense = _centers(n_frames, dense_slices, gap, center_offset=center_offset, jitter=0)
    extras = _extra_centers(base, dense, dense_slices - base_slices)
    combined = np.concatenate([base, extras]).astype(np.int64, copy=False)
    denom = float(max(n_frames - 1, 1))
    normalized_position = combined.astype(np.float32) / denom
    return (combined, normalized_position)

In [ ]:
def _native_center_crop(triplets: np.ndarray, fraction: float) -> np.ndarray:
    x = np.asarray(triplets)
    if x.ndim != 4:
        raise ValueError(f'established recipe expected [S,C,H,W], got {x.shape}')
    h, w = (int(x.shape[-2]), int(x.shape[-1]))
    crop_h = max(2, min(h, int(round(h * float(fraction)))))
    crop_w = max(2, min(w, int(round(w * float(fraction)))))
    top = (h - crop_h) // 2
    left = (w - crop_w) // 2
    return x[..., top:top + crop_h, left:left + crop_w]

In [ ]:
def constant_area_shape(height: int, width: int, *, reference_area: int=448**2, alignment: int=32) -> dict[str, int | float]:
    """Return isotropic resized and minimally stride-aligned rectangular geometry."""
    h, w = (int(height), int(width))
    area = int(reference_area)
    stride = int(alignment)
    if h < 1 or w < 1:
        raise ValueError('established recipe requires non-empty in-plane dimensions')
    if area < 4:
        raise ValueError('established recipe reference area must be positive')
    if stride < 1:
        raise ValueError('established recipe alignment must be positive')
    scale = math.sqrt(float(area) / float(h * w))
    resized_h = max(2, int(round(h * scale)))
    resized_w = max(2, int(round(w * scale)))
    aligned_h = int(math.ceil(resized_h / stride) * stride)
    aligned_w = int(math.ceil(resized_w / stride) * stride)
    pad_h = aligned_h - resized_h
    pad_w = aligned_w - resized_w
    top = pad_h // 2
    bottom = pad_h - top
    left = pad_w // 2
    right = pad_w - left
    return {'source_height': h, 'source_width': w, 'scale': float(scale), 'resized_height': resized_h, 'resized_width': resized_w, 'aligned_height': aligned_h, 'aligned_width': aligned_w, 'pad_top': top, 'pad_bottom': bottom, 'pad_left': left, 'pad_right': right, 'anatomical_pixels': int(resized_h * resized_w), 'tensor_pixels': int(aligned_h * aligned_w)}


def resize_triplets_constant_area(triplets: np.ndarray, *, reference_area: int=448**2, alignment: int=32) -> torch.Tensor:
    """Resize [S,C,H,W] once with one scale and reflection-pad only to stride."""
    x = np.asarray(triplets, dtype=np.float32)
    if x.ndim != 4:
        raise ValueError(f'established recipe expected [S,C,H,W], got {x.shape}')
    if int(x.shape[1]) != 3:
        raise ValueError('established recipe requires three-channel 2.5D triplets')
    geometry = constant_area_shape(int(x.shape[-2]), int(x.shape[-1]), reference_area=int(reference_area), alignment=int(alignment))
    tensor = torch.from_numpy(np.ascontiguousarray(x))
    resized = F.interpolate(tensor, size=(int(geometry['resized_height']), int(geometry['resized_width'])), mode='bilinear', align_corners=False, antialias=True)
    padding = (int(geometry['pad_left']), int(geometry['pad_right']), int(geometry['pad_top']), int(geometry['pad_bottom']))
    if any(padding):
        if max(padding[:2]) >= resized.shape[-1] or max(padding[2:]) >= resized.shape[-2]:
            raise ValueError('established recipe reflection padding is invalid for this resized geometry')
        resized = F.pad(resized, padding, mode='reflect')
    return resized


def preprocess_triplets(raw: np.ndarray, *, gap: int=1, center_offset: int=0, crop_fraction: float=0.90) -> tuple[torch.Tensor, np.ndarray]:
    """Return 32 native-aspect established recipe triplets at approximately constant pixel area."""
    if int(gap) < 1:
        raise ValueError('established recipe 2.5D gap must be positive')
    normalized = _normalise_volume(raw)
    centers, position = dense_centers(len(normalized), gap=int(gap), center_offset=int(center_offset))
    offsets = np.asarray([-int(gap), 0, int(gap)], dtype=np.int64)
    index = np.clip(centers[:, None] + offsets[None, :], 0, len(normalized) - 1)
    triplets = normalized[index].astype(np.float32, copy=False)
    cropped = _native_center_crop(triplets, float(crop_fraction))
    images = resize_triplets_constant_area(cropped)
    return (images, position)

In [ ]:
class KneeMRIDataset(Dataset):
    """One item is a complete study, with a variable number of rectangular series."""
    def __init__(self, uids, series_records, data_root, targets, weights):
        self.study_uids = list(uids)
        self.series_records = series_records
        self.data_root = Path(data_root)
        self.targets = np.asarray(targets, np.float32)
        self.weights = np.asarray(weights, np.float32)
        if self.targets.shape != (len(uids), N_TARGETS) or self.weights.shape != self.targets.shape:
            raise ValueError("Study supervision dimensions disagree.")
        if any(not series_records.get(uid) for uid in uids):
            raise ValueError("A study has no eligible series.")

    def __len__(self):
        return len(self.study_uids)

    def __getitem__(self, idx):
        uid = self.study_uids[idx]
        volumes, positions, meta, geometry = [], [], [], []
        for record in self.series_records[uid]:
            directory = find_series_dir(self.data_root, "train", uid, record["series_uid"])
            if directory is None:
                raise FileNotFoundError(f"Missing series {uid}/{record['series_uid']}")
            raw = read_dicom_series(directory)
            image, position = preprocess_triplets(raw)
            volumes.append(image)
            positions.append(torch.from_numpy(position))
            meta.append([record["plane_id"], record["fluid_id"], record["fat_id"]])
            geometry.append(dict(series_uid=record["series_uid"], height=image.shape[-2],
                                 width=image.shape[-1], present=True))
        return dict(study_uid=uid, volumes=volumes, slice_position=torch.stack(positions),
                    present=torch.ones(len(volumes)), series_meta=torch.tensor(meta, dtype=torch.long),
                    geometry=geometry, target=torch.from_numpy(self.targets[idx]),
                    weight=torch.from_numpy(self.weights[idx]))


def make_dataset(run_root, p, split):
    root = Path(run_root) / "protocol"
    index = load_json(root / "series_index.json")
    with np.load(root / "labels.npz", allow_pickle=False) as f:
        if split == "expert":
            uids, raw = f["expert_uids"].tolist(), f["expert_target"].copy()
            target, weight = np.nan_to_num(raw, nan=0.5), np.isfinite(raw).astype(np.float32)
        else:
            uids = p["splits"][split]
            lookup = {uid: i for i, uid in enumerate(f["uids"].tolist())}
            ix = [lookup[uid] for uid in uids]
            target, weight = f["target"][ix], f["weight"][ix]
    return KneeMRIDataset(uids, index, p["data_root"], target, weight)


def collate_studies(items):
    return list(items)


def make_loader(dataset, *, epoch=None):
    seed = RECIPE["seed"] + (0 if epoch is None else int(epoch) * 1009)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=RECIPE["batch_size"] if epoch is not None else 1,
                      shuffle=epoch is not None, drop_last=False, collate_fn=collate_studies,
                      num_workers=0, pin_memory=False, generator=generator)

## 6. DINOv2 and the complete study model

The public ViT-S/14 encoder turns each triplet into a **384-value vector**.
384 is the feature width, not the image size or the number of slices. Inputs
receive at most thirteen pixels of right/bottom reflection padding to fit
14×14 patches, followed by ImageNet channel normalization. The three channels
are neighboring grayscale MRI slices, not color channels.

A learned summary token and the slice vectors enter one transformer layer
with six attention heads, a 768-unit feed-forward block and dropout 0.1.
There is **no additional slice-position or physical-spacing embedding**.
Although vectors are sorted by their supplied position, this block provides
set context rather than explicit distance/direction reasoning.

Plane, fluid-sensitivity and fat-suppression embeddings describe each series.
Twelve learned finding queries attend over all series. A residual connection,
layer normalization and twelve finding-specific linear heads produce logits.
Every encoder layer and every new head is trainable. Gradient checkpointing
recomputes encoder activations during backward to save GPU memory.

In [ ]:
def state_digest(state):
    h = hashlib.sha256()
    for name, tensor in sorted(state.items()):
        tensor = tensor.detach().cpu().contiguous()
        h.update(name.encode())
        h.update(str(tensor.dtype).encode())
        h.update(str(tuple(tensor.shape)).encode())
        h.update(tensor.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()


def dino_backbone(*, pretrained=False):
    import timm
    return timm.create_model(DINO_MODEL, pretrained=pretrained, num_classes=0, dynamic_img_size=True, pretrained_cfg_overlay={'hf_hub_id': '', 'url': DINO_URL})


@dataclass
class SliceOutput:
    logits: torch.Tensor


class DinoSliceTransformer(nn.Module):
    """Encode triplets, pool slices within each series, and attend across series."""

    def __init__(self, encoder, *, dim=384, heads=6, dropout=0.1, chunk_size=2, gradient_checkpointing=True):
        super().__init__()
        if dim % heads or chunk_size < 1:
            raise ValueError('invalid slice transformer dimensions/chunk size')
        self.encoder = encoder
        self.encoder.requires_grad_(True)
        self.chunk_size = int(chunk_size)
        self.gradient_checkpointing = bool(gradient_checkpointing)
        self.series_cls = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(dim, heads, dim_feedforward=dim * 2, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.slice_context = nn.TransformerEncoder(layer, 1, norm=nn.LayerNorm(dim), enable_nested_tensor=False)
        self.plane = nn.Embedding(4, dim, padding_idx=0)
        self.fluid = nn.Embedding(3, dim, padding_idx=0)
        self.fat = nn.Embedding(3, dim, padding_idx=0)
        self.queries = nn.Parameter(torch.randn(1, N_TARGETS, dim) * 0.02)
        self.study_attention = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.classifier = nn.Parameter(torch.randn(N_TARGETS, dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(N_TARGETS))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406])[None, :, None, None])
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225])[None, :, None, None])

    def encode_chunk(self, x):
        ph, pw = (-x.shape[-2] % 14, -x.shape[-1] % 14)
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), mode='reflect')
        return self.encoder((x - self.mean.to(x.dtype)) / self.std.to(x.dtype))

    def forward(self, volumes, present, series_meta, slice_position):
        present = present.reshape(-1)
        meta = series_meta.reshape(-1, 3)
        position = slice_position.reshape(len(volumes), -1)
        if len(volumes) != len(present) or len(meta) != len(volumes):
            raise ValueError('established recipe series/mask/metadata mismatch')
        tokens = []
        for i, volume in enumerate(volumes):
            if float(present[i]) <= 0:
                continue
            if volume.ndim != 4 or volume.shape[1] != 3 or len(volume) != position.shape[1]:
                raise ValueError('established recipe expects [slices,3,H,W] and matching positions')
            chunks = []
            for x in volume.split(self.chunk_size):
                if self.training and self.gradient_checkpointing:
                    z = checkpoint(self.encode_chunk, x, use_reentrant=False, preserve_rng_state=True)
                else:
                    z = self.encode_chunk(x)
                chunks.append(z)
            z = torch.cat(chunks)[torch.argsort(position[i], stable=True)].unsqueeze(0)
            cls = self.series_cls.to(z.dtype)
            token = self.slice_context(torch.cat((cls, z), dim=1))[:, 0]
            m = meta[i]
            token = token + self.plane(m[0]) + self.fluid(m[1]) + self.fat(m[2])
            tokens.append(token)
        if not tokens:
            raise ValueError('established recipe study has no readable series')
        series = torch.stack(tokens, dim=1)
        query = self.queries.to(series.dtype)
        attended, _ = self.study_attention(query, series, series, need_weights=False)
        features = self.norm(query + attended)
        logits = (features * self.classifier[None]).sum(-1) + self.bias
        return SliceOutput(logits.float())

In [ ]:
def prepare_public_weights(run_root):
    root = Path(run_root) / "public_init"
    root.mkdir(parents=True, exist_ok=True)
    path, manifest_path = root / "encoder.pt", root / "encoder.json"
    if manifest_path.exists():
        return read_public_weights(run_root)[1]
    if path.exists():
        raise FileExistsError("Public weights have no manifest; inspect this incomplete initialization.")
    encoder = dino_backbone(pretrained=True)
    try:
        fingerprint = state_digest(encoder.state_dict())
        if fingerprint != PUBLIC_TENSOR_SHA256:
            raise ValueError("Downloaded weights differ from the pinned authors' public initialization.")
        atomic_torch_save(path, encoder.state_dict())
    finally:
        del encoder
        gc.collect()
    manifest = dict(source_url=DINO_URL, tensor_sha256=fingerprint,
                    sha256=sha256_file(path), competition_training_studies=[])
    write_json(manifest_path, manifest)
    return manifest


def read_public_weights(run_root):
    root = Path(run_root) / "public_init"
    manifest = load_json(root / "encoder.json")
    if (manifest["source_url"] != DINO_URL or manifest["tensor_sha256"] != PUBLIC_TENSOR_SHA256
            or manifest["competition_training_studies"] != []
            or manifest["sha256"] != sha256_file(root / "encoder.pt")):
        raise ValueError("Public initialization manifest/hash mismatch.")
    weights = torch.load(root / "encoder.pt", map_location="cpu", weights_only=True)
    if state_digest(weights) != PUBLIC_TENSOR_SHA256:
        raise ValueError("Public initialization tensors changed.")
    return weights, manifest


def build_model(run_root=None):
    encoder = dino_backbone(pretrained=False)
    if run_root is not None:
        encoder.load_state_dict(read_public_weights(run_root)[0], strict=True)
    return DinoSliceTransformer(encoder, dim=RECIPE["dim"], heads=RECIPE["heads"],
                               dropout=RECIPE["dropout"], chunk_size=RECIPE["encoder_chunk_size"],
                               gradient_checkpointing=RECIPE["gradient_checkpointing"])


def parameter_groups(model):
    encoder = list(model.encoder.parameters())
    if not encoder or not all(p.requires_grad for p in encoder):
        raise ValueError("Every encoder layer must be trainable.")
    ids = {id(p) for p in encoder}
    heads = [p for p in model.parameters() if p.requires_grad and id(p) not in ids]
    groups = [dict(params=encoder, lr=RECIPE["encoder_lr"], name="public_encoder"),
              dict(params=heads, lr=RECIPE["head_lr"], name="fresh_heads")]
    grouped = [id(p) for g in groups for p in g["params"]]
    if len(grouped) != len(set(grouped)) or set(grouped) != {id(p) for p in model.parameters() if p.requires_grad}:
        raise RuntimeError("Optimizer must cover trainable parameters exactly once.")
    return groups

## 7. Probabilities, loss and AUC

Each finding has its own logit z. Its probability is sigmoid(z) = 1/(1+exp(-z)).
For example, logits −2, 0 and 2 become about 0.119, 0.5 and 0.881. Findings
are independent outputs: probabilities do not need to add to one. A sigmoid
output alone does not establish clinical calibration.

Training uses numerically stable binary cross-entropy with logits. Label
weights mask uncertain cells; inverse target weight-mass balances the twelve
findings. Two studies form one optimizer update, processed sequentially and
scaled by their supervised weight mass. This matches the weighted loss of the
combined batch. Even zero-weight studies remain in MRI exposure.

Evaluation computes each finding's ROC AUC on cells with weight greater than
zero, with report targets above 0.5 treated as positive. AUC itself is
unweighted. Macro AUC averages defined findings; all twelve must be measurable
on scanner validation. An undefined expert AUC is reported as missing.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def target_balance_multipliers(weights: np.ndarray) -> np.ndarray:
    weights = np.asarray(weights, dtype=np.float64)
    if weights.ndim != 2 or weights.shape[1] != len(TARGETS):
        raise ValueError('weights must have shape [N,12]')
    mass = weights.sum(axis=0)
    valid = mass > 0
    if not valid.all():
        missing = [TARGETS[j] for j in np.flatnonzero(~valid)]
        raise ValueError(f'established recipe has no usable supervision for target(s): {missing}')
    mean_mass = float(mass.mean())
    return (mean_mass / mass).astype(np.float32)


def target_balanced_weak_bce(logits: torch.Tensor, target: torch.Tensor, weight: torch.Tensor, target_multiplier: torch.Tensor) -> torch.Tensor:
    if logits.shape != target.shape or logits.shape != weight.shape:
        raise ValueError('logits/target/weight shapes must match')
    if logits.ndim != 2 or logits.shape[1] != len(TARGETS):
        raise ValueError('established recipe logits must have shape [B,12]')
    multiplier = torch.as_tensor(target_multiplier, dtype=logits.dtype, device=logits.device)
    if multiplier.shape != (len(TARGETS),):
        raise ValueError('target_multiplier must have shape [12]')
    effective = weight * multiplier[None, :]
    denominator = effective.sum()
    if float(denominator.detach().item()) <= 0:
        return logits.sum() * 0.0
    cell = nn.functional.binary_cross_entropy_with_logits(logits, target, reduction='none')
    return (cell * effective).sum() / denominator.clamp_min(1e-08)

In [ ]:
def _move_study(item: dict, device) -> tuple:
    volumes = [volume.to(device, non_blocking=True) for volume in item['volumes']]
    return (volumes, item['slice_position'].to(device, non_blocking=True), item['present'].to(device, non_blocking=True), item['series_meta'].to(device, non_blocking=True), item['target'].to(device, non_blocking=True).unsqueeze(0), item['weight'].to(device, non_blocking=True).unsqueeze(0))


def _study_mass(weight: torch.Tensor, target_multiplier: torch.Tensor) -> float:
    w = weight.reshape(-1, weight.shape[-1]).to(dtype=torch.float32, device='cpu')
    m = target_multiplier.to(dtype=torch.float32, device='cpu')
    return float((w * m[None, :]).sum().item())


def _batch_scales(items: list[dict], multiplier_cpu: torch.Tensor) -> list[float]:
    masses = [_study_mass(item['weight'], multiplier_cpu) for item in items]
    total = float(sum(masses))
    if total > 0:
        return [float(mass / total) for mass in masses]
    return [1.0 / len(items)] * len(items)

In [ ]:
def macro_auc(target, weight, prediction):
    target, weight, prediction = map(np.asarray, (target, weight, prediction))
    if target.shape != weight.shape or target.shape != prediction.shape or target.shape[1] != N_TARGETS:
        raise ValueError("AUC inputs must have matching [studies,12] shapes.")
    if not all(np.isfinite(a).all() for a in (target, weight, prediction)):
        raise ValueError("Nonfinite AUC input.")
    scores = {}
    for j, name in enumerate(TARGETS):
        active = weight[:, j] > 0
        truth = (target[active, j] > 0.5).astype(int)
        scores[name] = float(roc_auc_score(truth, prediction[active, j])) if len(np.unique(truth)) == 2 else float("nan")
    defined = [v for v in scores.values() if np.isfinite(v)]
    return dict(macro_auc=float(np.mean(defined)) if defined else float("nan"),
                per_target_auc=scores, targets_defined=len(defined))


def loss_for(model, item, device, amp_dtype, multiplier):
    volumes, position, present, meta, target, weight = _move_study(item, device)
    with autocast(device, amp_dtype):
        output = model(volumes, present, meta, position)
        loss = target_balanced_weak_bce(output.logits, target, weight, multiplier)
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Nonfinite loss: {item['study_uid']}")
    return output, loss


def batch_step(model, items, device, amp_dtype, optimizer, scaler, multiplier):
    optimizer.zero_grad(set_to_none=True)
    total = 0.0
    for item, scale in zip(items, _batch_scales(items, multiplier)):
        output, loss = loss_for(model, item, device, amp_dtype, multiplier.to(device))
        scaler.scale(loss * scale).backward()
        total += float(loss.detach()) * scale
        del output, loss
    scaler.unscale_(optimizer)
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), RECIPE["grad_clip"], error_if_nonfinite=True)
    scaler.step(optimizer)
    scaler.update()
    return total, float(grad_norm)


@torch.no_grad()
def predict(model, loader, device, amp_dtype):
    was_training = model.training
    uids, probabilities, targets, weights = [], [], [], []
    model.eval()
    try:
        for items in loader:
            for item in items:
                volumes, position, present, meta, _, _ = _move_study(item, device)
                with autocast(device, amp_dtype):
                    output = model(volumes, present, meta, position)
                probability = output.logits.float().sigmoid().cpu().numpy().reshape(-1)
                if not np.isfinite(probability).all():
                    raise FloatingPointError(f"Nonfinite prediction: {item['study_uid']}")
                uids.append(item["study_uid"])
                probabilities.append(probability)
                targets.append(item["target"].numpy())
                weights.append(item["weight"].numpy())
                del output, volumes
    finally:
        model.train(was_training)
    if not uids or len(uids) != len(set(uids)):
        raise ValueError("Empty or duplicate prediction UIDs.")
    return dict(uids=np.asarray(uids), prediction=np.stack(probabilities),
                target=np.stack(targets), weight=np.stack(weights))

## 8. Preflight, training and recovery

The real-data preflight selects the two training studies with the most eligible
series, runs backward, and verifies finite nonzero gradients in the encoder
and new heads. It performs **zero optimizer updates** and discards that model.
It estimates peak GPU allocation for those studies, not every possible scan.

Training uses AdamW with encoder learning rate 0.00001, head learning rate
0.0001, weight decay 0.0001 and gradient clipping at 1.0. A cosine schedule
reaches one percent of each initial learning rate after twelve epochs.
The seed is 2026 and shuffling is reconstructed deterministically per epoch.
On a compatible 5090, automatic mixed precision uses bfloat16.

The checkpoint includes optimizer, scheduler, gradient scaler, random-number
states and complete history. Evaluation uses the **fixed final epoch**, not
the epoch with the highest observed validation score. Training runs directly
in this kernel; keep Jupyter alive. Ordinary interruption cleans up the model.

In [ ]:
def rng_state() -> dict:
    """Every generator that can change what the next epoch does."""
    state = {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state(), 'cuda': None}
    if torch.cuda.is_available():
        state['cuda'] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict | None) -> dict:
    """Restore what is present and report what was skipped, rather than raising.

    A run that moves between machines will not have the same device count, and
    refusing to resume over that would cost more than the exactness is worth.
    The caller is told, so a partial restore is never silent.
    """
    skipped: list[str] = []
    if not state:
        return {'restored': [], 'skipped': ['all']}
    restored: list[str] = []
    if state.get('python') is not None:
        random.setstate(state['python'])
        restored.append('python')
    if state.get('numpy') is not None:
        np.random.set_state(state['numpy'])
        restored.append('numpy')
    if state.get('torch') is not None:
        torch.set_rng_state(torch.as_tensor(state['torch'], dtype=torch.uint8))
        restored.append('torch')
    cuda = state.get('cuda')
    if cuda is None:
        skipped.append('cuda')
    elif not torch.cuda.is_available():
        skipped.append('cuda: no device')
    elif len(cuda) != torch.cuda.device_count():
        skipped.append(f'cuda: saved on {len(cuda)} device(s), running on {torch.cuda.device_count()}')
    else:
        torch.cuda.set_rng_state_all([torch.as_tensor(x, dtype=torch.uint8) for x in cuda])
        restored.append('cuda')
    return {'restored': restored, 'skipped': skipped}

In [ ]:
def run_contract(run_root, device, amp_dtype):
    p = load_protocol(run_root)
    _, manifest = read_public_weights(run_root)
    return dict(implementation=p["implementation"],
                protocol_sha256=sha256_file(Path(run_root) / "protocol/protocol.json"),
                public_encoder_sha256=manifest["sha256"], public_source_url=DINO_URL,
                training_uids_sha256=p["split_uid_hashes"]["train"],
                validation_uids_sha256=p["split_uid_hashes"]["validation"],
                competition_supervised_ancestor=False, gold_studies_in_gradient=0,
                epochs=RECIPE["epochs"], precision=str(amp_dtype), device_type=device.type,
                torch=str(torch.__version__), torchvision=str(torchvision.__version__),
                timm=timm.__version__, numpy=np.__version__)


def release_accelerator():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def preflight(run_root, device="cuda:0"):
    root = Path(run_root)
    model = groups = output = loss = None
    with exclusive_run(root):
        claim_run(root)
        p = load_protocol(root)
        device, amp_dtype = runtime_for(device)
        contract = run_contract(root, device, amp_dtype)
        seed_everything(RECIPE["seed"])
        try:
            model = build_model(root).to(device).train()
            groups = parameter_groups(model)
            dataset = make_dataset(root, p, "train")
            multiplier = torch.from_numpy(target_balance_multipliers(dataset.weights)).to(device)
            active = [i for i, w in enumerate(dataset.weights) if w.sum() > 0]
            chosen = sorted(active, key=lambda i: (-len(dataset.series_records[dataset.study_uids[i]]),
                                                  dataset.study_uids[i]))[:RECIPE["batch_size"]]
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(device)
            for i in chosen:
                output, loss = loss_for(model, dataset[i], device, amp_dtype, multiplier)
                (loss / len(chosen)).backward()
                output = loss = None
            counts = {}
            for group in groups:
                grads = [p.grad for p in group["params"] if p.grad is not None]
                if not grads or any(not torch.isfinite(g).all() for g in grads):
                    raise RuntimeError(f"Invalid gradients: {group['name']}")
                count = sum(int(torch.count_nonzero(g) > 0) for g in grads)
                if not count:
                    raise RuntimeError(f"No learning signal: {group['name']}")
                counts[group["name"]] = count
            encoded = list(model.encoder.parameters())
            if any(x.grad is None or not torch.count_nonzero(x.grad) for x in (encoded[0], encoded[-1])):
                raise RuntimeError("Gradient did not reach both ends of the image encoder.")
            result = dict(passed=True, contract=contract, optimizer_steps=0, gradient_tensor_counts=counts,
                          study_uids=[dataset.study_uids[i] for i in chosen],
                          peak_cuda_gib=torch.cuda.max_memory_allocated(device)/2**30 if device.type == "cuda" else None)
            write_json(root / "preflight.json", result)
            print("Model preflight: PASS; optimizer updates: 0; peak CUDA GiB:", result["peak_cuda_gib"])
            return result
        except BaseException as error:
            import traceback
            traceback.clear_frames(error.__traceback__)
            raise
        finally:
            model = groups = output = loss = None
            # The local parameter/gradient lists also own GPU storage.
            encoded = grads = group = multiplier = None
            release_accelerator()


def save_recovery(path, epoch, model, optimizer, scheduler, scaler, history, contract):
    atomic_torch_save(path, dict(contract=contract, epoch=epoch, model_state=model.state_dict(),
                                optimizer_state=optimizer.state_dict(), scheduler_state=scheduler.state_dict(),
                                scaler_state=scaler.state_dict(), rng_state=rng_state(), history=history))


def save_predictions(path, prediction, contract, split, checkpoint_sha):
    path = Path(path)
    atomic_npz(path, **prediction)
    write_json(path.with_suffix(".json"), dict(contract=contract, split=split,
               checkpoint_sha256=checkpoint_sha, npz_sha256=sha256_file(path),
               uid_sha256=digest(prediction["uids"].tolist())))


def train_model(run_root, device="cuda:0"):
    root, out = Path(run_root), Path(run_root) / "model"
    model = optimizer = scheduler = scaler = groups = saved = None
    with exclusive_run(root):
        claim_run(root)
        p = load_protocol(root)
        device, amp_dtype = runtime_for(device)
        contract = run_contract(root, device, amp_dtype)
        check = load_json(root / "preflight.json")
        require_same_run(check["contract"], contract)
        if check.get("passed") is not True or check.get("optimizer_steps") != 0:
            raise ValueError("Run and pass the real-data preflight first.")
        if (out / "complete.json").exists():
            require_same_run(load_json(out / "complete.json")["contract"], contract)
            evaluate_results(root)
            print("Training is already complete; final artifacts verified.")
            return out / "final.pt"
        recovery = out / "recovery_latest.pt"
        if out.exists() and any(p.name != "recovery_latest.pt.writing" for p in out.iterdir()) and not recovery.exists():
            raise FileExistsError("Model directory is occupied without a recovery checkpoint.")
        seed_everything(RECIPE["seed"])
        try:
            model = build_model(root).to(device).train()
            groups = parameter_groups(model)
            optimizer = torch.optim.AdamW(groups, weight_decay=RECIPE["weight_decay"])
            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
                lambda e: .01 + .99 * (1 + np.cos(np.pi * min(e, RECIPE["epochs"]) / RECIPE["epochs"])) / 2)
            scaler = torch.amp.GradScaler(device.type, enabled=amp_dtype is torch.float16)
            training = make_dataset(root, p, "train")
            validation = make_dataset(root, p, "validation")
            multiplier = torch.from_numpy(target_balance_multipliers(training.weights))
            history, start_epoch = [], 1
            if recovery.exists():
                # Trusted local checkpoint written by this notebook, includes Python/NumPy RNG state.
                saved = torch.load(recovery, map_location="cpu", weights_only=False)
                require_same_run(saved["contract"], contract)
                if not 0 < saved["epoch"] <= RECIPE["epochs"]:
                    raise ValueError("Invalid recovery epoch.")
                model.load_state_dict(saved["model_state"], strict=True)
                optimizer.load_state_dict(saved["optimizer_state"])
                scheduler.load_state_dict(saved["scheduler_state"])
                scaler.load_state_dict(saved["scaler_state"])
                set_rng_state(saved["rng_state"])
                history, start_epoch = list(saved["history"]), saved["epoch"] + 1
                if [row["epoch"] for row in history] != list(range(1, start_epoch)):
                    raise ValueError("Recovery history does not match completed epochs.")
                saved = None
            else:
                seed_everything(RECIPE["seed"] + 101)
            print(f"Training on {len(training)} studies; starting epoch {start_epoch} of {RECIPE['epochs']}.")
            for epoch in range(start_epoch, RECIPE["epochs"] + 1):
                start = time.monotonic()
                loader = make_loader(training, epoch=epoch)
                model.train()
                loss_sum, seen = 0.0, []
                for batch, items in enumerate(loader, 1):
                    loss, grad = batch_step(model, items, device, amp_dtype, optimizer, scaler, multiplier)
                    loss_sum += loss
                    seen.extend(item["study_uid"] for item in items)
                    if batch % 100 == 0:
                        print(f"Epoch {epoch}: {batch}/{len(loader)} batches; loss={loss_sum / batch:.5f}", flush=True)
                if len(seen) != len(set(seen)) or set(seen) != set(p["splits"]["train"]):
                    raise RuntimeError("Epoch did not expose exactly the permitted training studies.")
                del loader
                scheduler.step()
                predictions = predict(model, make_loader(validation), device, amp_dtype)
                scores = macro_auc(predictions["target"], predictions["weight"], predictions["prediction"])
                if scores["targets_defined"] != N_TARGETS or not np.isfinite(scores["macro_auc"]):
                    raise RuntimeError("Validation must define twelve AUCs.")
                history.append(dict(epoch=epoch, train_loss=loss_sum / batch, validation=scores,
                                    training_uids_sha256=digest(sorted(seen)), epoch_minutes=(time.monotonic()-start)/60,
                                    learning_rates=[float(g["lr"]) for g in optimizer.param_groups]))
                save_recovery(recovery, epoch, model, optimizer, scheduler, scaler, history, contract)
                write_json(out / "history.json", history)
                print(f"Epoch {epoch} complete: macro AUC={scores['macro_auc']:.6f}", flush=True)
            encoder_sha = state_digest(model.encoder.state_dict())
            if encoder_sha == PUBLIC_TENSOR_SHA256:
                raise RuntimeError("The public encoder was not updated during training.")
            final = out / "final.pt"
            atomic_torch_save(final, dict(contract=contract, completed_epochs=RECIPE["epochs"],
                                         selection="fixed_final_epoch", recipe=RECIPE,
                                         model_state=model.state_dict(), history=history,
                                         encoder_tensor_sha256_final=encoder_sha))
            # A crash after the last recovery can leave history.json behind by one epoch.
            write_json(out / "history.json", history)
            checkpoint_sha = sha256_file(final)
            for split, dataset in (("validation", validation), ("expert", make_dataset(root, p, "expert"))):
                predictions = predict(model, make_loader(dataset), device, amp_dtype)
                save_predictions(out / f"{split}.npz", predictions, contract, split, checkpoint_sha)
            write_json(out / "complete.json", dict(contract=contract, checkpoint_sha256=checkpoint_sha,
                                                   completed_epochs=RECIPE["epochs"], expert_role="diagnostic only"))
            print("Training and fixed-final-epoch predictions: COMPLETE", flush=True)
            return final
        except BaseException as error:
            import traceback
            traceback.clear_frames(error.__traceback__)
            raise
        finally:
            model = optimizer = scheduler = scaler = groups = saved = None
            release_accelerator()

## 9. Verified evaluation and reusable inference

Before exporting scores, evaluation checks the checkpoint hash, prediction
hashes, exact study lists, labels and confidence masks. The final model can
also be reconstructed entirely from the notebook definitions and its saved
state dictionary. It needs no repository checkpoint loader and performs no
public-weight download when loading a trained checkpoint.

In [ ]:
def evaluate_results(run_root):
    root, out = Path(run_root), Path(run_root) / "model"
    p = load_protocol(root)
    complete = load_json(out / "complete.json")
    contract = complete["contract"]
    if (contract["implementation"] != p["implementation"]
            or contract["protocol_sha256"] != sha256_file(root / "protocol/protocol.json")
            or complete["completed_epochs"] != RECIPE["epochs"]
            or contract["epochs"] != RECIPE["epochs"]
            or contract["competition_supervised_ancestor"] is not False
            or contract["gold_studies_in_gradient"] != 0):
        raise ValueError("Final model/protocol contract mismatch.")
    checkpoint_sha = sha256_file(out / "final.pt")
    if checkpoint_sha != complete["checkpoint_sha256"]:
        raise ValueError("Final checkpoint changed.")
    metrics, tables, exports = {}, {}, {}
    for split in ("validation", "expert"):
        path = out / f"{split}.npz"
        meta = load_json(path.with_suffix(".json"))
        if (meta["contract"] != contract or meta["split"] != split
                or meta["checkpoint_sha256"] != checkpoint_sha or meta["npz_sha256"] != sha256_file(path)):
            raise ValueError(f"Prediction provenance mismatch: {split}")
        with np.load(path, allow_pickle=False) as f:
            data = {k: f[k].copy() for k in ("uids", "target", "weight", "prediction")}
        uids = data["uids"].tolist()
        expected = make_dataset(root, p, split)
        if len(uids) != len(set(uids)) or set(uids) != set(expected.study_uids) or digest(uids) != meta["uid_sha256"]:
            raise ValueError(f"Wrong prediction studies: {split}")
        lookup = {uid: i for i, uid in enumerate(uids)}
        ix = [lookup[uid] for uid in expected.study_uids]
        data = {k: v[ix] for k, v in data.items()}
        for key in ("target", "weight", "prediction"):
            if data[key].shape != expected.targets.shape or not np.isfinite(data[key]).all():
                raise ValueError(f"Invalid prediction array: {split}/{key}")
        if not np.array_equal(data["target"], expected.targets) or not np.array_equal(data["weight"], expected.weights):
            raise ValueError(f"Prediction labels or masks changed: {split}")
        if ((data["prediction"] < 0) | (data["prediction"] > 1)).any():
            raise ValueError("Probabilities must be in [0,1].")
        scores = macro_auc(data["target"], data["weight"], data["prediction"])
        if split == "validation" and scores["targets_defined"] != N_TARGETS:
            raise ValueError("Validation must define all twelve AUCs.")
        table = pd.DataFrame.from_dict(coverage(data["target"], data["weight"]), orient="index")
        table["AUC"] = pd.Series(scores["per_target_auc"])
        metrics[split], tables[split] = finite_json(scores), table
        exports[split] = pd.DataFrame(data["prediction"], columns=TARGETS).assign(StudyInstanceUID=data["uids"])[["StudyInstanceUID", *TARGETS]]
    reports = root / "reports"
    reports.mkdir(exist_ok=True)
    write_json(reports / "metrics.json", metrics)
    for split in tables:
        tables[split].to_csv(reports / f"{split}_by_finding.csv", index_label="Finding")
        exports[split].to_csv(reports / f"{split}_probabilities.csv", index=False)
    return metrics, tables


def load_trained_model(checkpoint_path, device="cuda:0"):
    """Load a trusted checkpoint created by this standalone notebook."""
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if payload["recipe"] != RECIPE or payload["completed_epochs"] != RECIPE["epochs"]:
        raise ValueError("Checkpoint recipe/endpoint mismatch.")
    require_same_run(payload["contract"]["implementation"], implementation_contract())
    model = build_model().to(device)
    model.load_state_dict(payload["model_state"], strict=True)
    return model.eval()

## 10. Plotting functions

The architecture and sigmoid figures are explanatory. Slice previews use
actual training images; learning curves and per-finding plots use saved
results from this run. PNG and SVG versions are saved alongside the run.

In [ ]:
def architecture():
    """Exact default architecture; arrows show tensors, not training history."""
    fig, ax = plt.subplots(figsize=(11, 12), layout='constrained')
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.axis('off')
    ax.text(0.04, 0.985, 'Knee MRI → twelve finding probabilities', fontsize=19, weight='bold', va='top', color='#15334a')
    ax.text(0.04, 0.95, 'DINOv2 image features · slice context · attention across MRI series', fontsize=11, va='top', color='#526777')

    def box(x, y, w, h, title, detail, color='#eaf2f8'):
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle='round,pad=0.008,rounding_size=0.008', facecolor=color, edgecolor='#8ba5b7', lw=1))
        ax.text(x, y + 0.012, title, ha='center', va='center', fontsize=11, weight='bold', color='#15334a')
        ax.text(x, y - 0.017, detail, ha='center', va='center', fontsize=9, color='#344e60')

    def arrow(start, end):
        ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=13, color='#526777', lw=1.4))
    centers = [0.87, 0.75, 0.63, 0.51, 0.39, 0.27, 0.15]
    stages = [('One study: K eligible MRI series', 'Each series keeps its own rectangular geometry'), ('32 three-slice inputs per series', '[32, 3, H, W] · 90% central crop · area ≈ 448²'), ('DINOv2 ViT-S/14 encoder', '14 × 14 image patches → [32, 384] slice features'), ('One series transformer', 'Summary token + 32 features → one 384-value vector'), ('Series vectors + scan metadata', 'K vectors · [K, 384]'), ('Attention across series', '12 learned finding queries · 6 attention heads'), ('Twelve finding-specific linear heads', '12 logits → elementwise sigmoid → 12 probabilities')]
    for i, (y, (title, detail)) in enumerate(zip(centers, stages)):
        box(0.345, y, 0.59, 0.075, title, detail, '#e7f4ee' if i == 6 else '#eaf2f8')
        if i:
            arrow((0.345, centers[i - 1] - 0.046), (0.345, y + 0.046))
    box(0.83, 0.39, 0.25, 0.09, 'Scan metadata', 'Plane / fluid / fat', '#fff2dd')
    arrow((0.697, 0.39), (0.65, 0.39))
    box(0.83, 0.27, 0.25, 0.09, 'Finding queries', '12 × 384 trainable values', '#fff2dd')
    arrow((0.697, 0.27), (0.65, 0.27))
    ax.text(0.7, 0.62, 'The encoder is fully\nfine-tuned from public\npretrained weights.', fontsize=10, color='#344e60', va='center', linespacing=1.6)
    ax.text(0.7, 0.51, 'No additional slice-\nposition or physical-\nspacing embedding.', fontsize=10, color='#344e60', va='center', linespacing=1.6)
    ax.text(0.04, 0.055, '384 is the feature-vector width, not an image size. K varies by study.\nThe three input channels are neighboring MRI slices, not RGB colors.', fontsize=11, color='#344e60', linespacing=1.7)
    return fig


def slice_example(item, *, series_index=0, center_index=None):
    volume = item['volumes'][series_index].detach().cpu().numpy()
    positions = item['slice_position'][series_index].detach().cpu().numpy()
    if center_index is None:
        center_index = len(volume) // 2
    if not 0 <= center_index < len(volume):
        raise IndexError('Choose a sampled center that exists in this series.')
    fig = plt.figure(figsize=(11, 6.5), layout='constrained')
    grid = fig.add_gridspec(2, 3, height_ratios=(5, 1))
    for channel, title in enumerate(('Previous slice', 'Center slice', 'Next slice')):
        ax = fig.add_subplot(grid[0, channel])
        ax.imshow(volume[center_index, channel], cmap='gray', vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')
    ax = fig.add_subplot(grid[1, :])
    ax.scatter(positions, np.zeros_like(positions), color='#397ab0', label='Sampled centers')
    ax.scatter([positions[center_index]], [0], color='#dd7b30', s=90, label='Displayed triplet', zorder=3)
    ax.set(xlim=(-0.03, 1.03), ylim=(-0.4, 0.4), yticks=[], xlabel='Normalized through-series position')
    ax.legend(loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.45))
    fig.suptitle(f'Actual training input · series {series_index + 1} · {tuple(volume.shape)}', weight='bold')
    return fig


def sigmoid():
    z = np.linspace(-7, 7, 400)
    p = 1 / (1 + np.exp(-z))
    fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
    ax.plot(z, p, color='#397ab0', lw=2)
    ax.scatter([-2, 0, 2], 1 / (1 + np.exp(-np.array([-2, 0, 2]))), color='#dd7b30', zorder=3)
    ax.axhline(0.5, color='#b2bec8', ls='--', lw=1)
    ax.set(xlabel='Logit z (raw model score)', ylabel='Sigmoid probability', ylim=(-0.02, 1.02), title='Sigmoid: a score becomes a number between 0 and 1')
    ax.grid(alpha=0.15)
    return fig


def training_history(history):
    epochs = [r['epoch'] for r in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    axes[0].plot(epochs, [r['train_loss'] for r in history], 'o-', color='#397ab0')
    axes[0].set(title='Training loss', xlabel='Completed epoch', ylabel='Weighted binary cross-entropy')
    axes[1].plot(epochs, [r['validation']['macro_auc'] for r in history], 'o-', color='#32947b')
    axes[1].set(title='Scanner validation AUC', xlabel='Completed epoch', ylabel='Macro ROC AUC', ylim=(0, 1))
    for ax in axes:
        ax.set_xticks(epochs)
        ax.grid(alpha=0.2)
    return fig


def target_auc(tables):
    findings = list(tables['validation'].index)
    x = np.arange(len(findings))
    fig, ax = plt.subplots(figsize=(11, 6), layout='constrained')
    ax.barh(x + 0.18, tables['validation'].loc[findings, 'AUC'], height=0.34, label='Scanner validation', color='#397ab0')
    ax.barh(x - 0.18, tables['expert'].loc[findings, 'AUC'], height=0.34, label='Expert diagnostic', color='#77b8a4')
    ax.axvline(0.5, ls='--', color='#a1abb3', lw=1)
    ax.set(yticks=x, yticklabels=findings, xlim=(0, 1), xlabel='ROC AUC', title='Fixed final model · AUC by finding')
    ax.invert_yaxis()
    ax.legend(loc='lower left')
    return fig


def save_figure(fig, run_root, name):
    from pathlib import Path
    out = Path(run_root) / 'figures'
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f'{name}.png', dpi=160, facecolor='white')
    fig.savefig(out / f'{name}.svg', facecolor='white')

In [ ]:
IMPLEMENTATION_NAMES = ['sha256_file', 'digest', 'write_json', 'coverage', 'require_same_run', 'atomic_torch_save', 'atomic_npz', 'finite_json', 'notebook_code_digest', 'implementation_contract', 'exclusive_run', 'claim_run', 'runtime_for', 'autocast', 'environment_check', '_sort_key', 'find_series_dir', '_iter_dicom_files', '_pixel_spacing', '_pad_or_crop', 'read_dicom_series', '_normalise_volume', '_centers', 'plane_from_orientation', 'weighting_from_parameters', 'is_fluid_sensitive', '_multiframe_orientation', 'read_series_metadata', '_require_columns', 'load_train_csv', 'gold_mask', 'coerce_bool', 'normalise_plane', 'load_series_csv', 'backfill_series_metadata', '_flag_id', 'build_variable_series_index', 'load_fill_merged_export', 'prepare_all_report_only_supervision', 'verify_selection_split', 'partition', 'only_match', 'load_json', 'discover_inputs', 'canonical_split_rows', 'validate_series_policy', 'prepare_data', 'load_protocol', '_extra_centers', 'dense_centers', '_native_center_crop', 'constant_area_shape', 'resize_triplets_constant_area', 'preprocess_triplets', 'KneeMRIDataset', 'make_dataset', 'collate_studies', 'make_loader', 'state_digest', 'dino_backbone', 'SliceOutput', 'DinoSliceTransformer', 'prepare_public_weights', 'read_public_weights', 'build_model', 'parameter_groups', 'seed_everything', 'target_balance_multipliers', 'target_balanced_weak_bce', '_move_study', '_study_mass', '_batch_scales', 'macro_auc', 'loss_for', 'batch_step', 'predict', 'rng_state', 'set_rng_state', 'run_contract', 'release_accelerator', 'preflight', 'save_recovery', 'save_predictions', 'train_model', 'evaluate_results', 'load_trained_model', 'architecture', 'slice_example', 'sigmoid', 'training_history', 'target_auc', 'save_figure']

## 11. Run the workflow

All definitions are now loaded. The following cells do the work. Review the
population table and preview, then let the preflight finish before starting
training. A repeated preparation verifies the same inputs. Repeating training
after an interruption resumes this notebook's last completed epoch.

In [ ]:
environment_check(DEVICE)
protocol = prepare_data(work_root=WORK_ROOT, data_root=DATA_ROOT, run_root=RUN_ROOT,
                        labels_root=LABELS_ROOT, scanner_split_root=SCANNER_SPLIT_ROOT,
                        series_policy=SERIES_POLICY)
with exclusive_run(RUN_ROOT):
    claim_run(RUN_ROOT)
    public_manifest = prepare_public_weights(RUN_ROOT)
population = {**protocol["counts"], "expert": len(protocol["expert_uids"])}
display(pd.DataFrame.from_dict(population, orient="index", columns=["Studies"]))
display(pd.DataFrame.from_dict(protocol["coverage"]["train"], orient="index"))

In [ ]:
example = make_dataset(RUN_ROOT, protocol, "train")[0]
display(pd.DataFrame(example["geometry"])[["height", "width", "present"]].rename_axis("Series"))
fig = slice_example(example)
save_figure(fig, RUN_ROOT, "slice_inputs")
plt.show()
plt.close(fig)
del example

for name, draw in (("architecture", architecture), ("sigmoid", sigmoid)):
    fig = draw()
    save_figure(fig, RUN_ROOT, name)
    plt.show()
    plt.close(fig)

In [ ]:
preflight_result = preflight(RUN_ROOT, DEVICE)

In [ ]:
final_checkpoint = train_model(RUN_ROOT, DEVICE)
print("Final model:", final_checkpoint)

## 12. Learning curves and final results

The loss curve shows the average weighted training loss. The AUC curve shows
scanner validation after each completed epoch. The final reports always use
the model after epoch twelve, even if an earlier point is higher.

The prediction CSVs contain one row per held-out study and twelve sigmoid
probabilities. Missing expert AUCs indicate insufficient class coverage; they
are not zero scores. These are evaluation exports, not competition submissions.

In [ ]:
history = load_json(RUN_ROOT / "model/history.json")
fig = training_history(history)
save_figure(fig, RUN_ROOT, "training_history")
plt.show()
plt.close(fig)

metrics, tables = evaluate_results(RUN_ROOT)
display(pd.DataFrame({name: {"Macro AUC": row["macro_auc"], "Defined findings": row["targets_defined"]}
                      for name, row in metrics.items()}).T)
for split, table in tables.items():
    print(split)
    display(table)
fig = target_auc(tables)
save_figure(fig, RUN_ROOT, "auc_by_finding")
plt.show()
plt.close(fig)
print("Saved checkpoint, probabilities, metrics and figures:", RUN_ROOT)

## Outputs and recovery

| Location under RUN_ROOT | Contents |
|---|---|
| notebook_run.json | Executed implementation fingerprint and recipe |
| protocol/ | Input hashes, frozen labels, study lists and series metadata |
| public_init/ | Verified public image-encoder initialization |
| preflight.json | Actual backward-pass checks and peak GPU allocation |
| model/recovery_latest.pt | Latest completed epoch, optimizer and random states |
| model/final.pt | Fixed final model and architecture recipe |
| model/history.json | Training loss and validation AUC by epoch |
| reports/ | Metrics, per-finding coverage and per-study probabilities |
| figures/ | Architecture, real MRI preview and result plots |

To resume, restart the kernel, retain the same code, settings and inputs, and
run the cells in order. A partial epoch is repeated; a completed run is
verified and returned. New code or a new recipe requires a new run directory.
The data and public initialization may be copied to another machine, but a
resume contract still requires matching file paths, software and precision.

To use the saved model, call load_trained_model with model/final.pt. Build
unseen-study items with KneeMRIDataset using the same preprocessing and series
metadata, then pass their DataLoader through predict. The helper returns the
independent sigmoid probabilities for the twelve findings.

References: [DINOv2 paper](https://arxiv.org/abs/2304.07193),
[authors' implementation](https://github.com/facebookresearch/dinov2),
[PyTorch binary cross-entropy with logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html),
[ROC AUC definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).